# PubChem global-retrieval figures (random split)

Top-k accuracy vs. k, one figure per ranking mode, ICICLE vs. NEIMS vs.
MassFormer. Candidates are the **entire PubChem database** (~87M molecules,
same molecular formula not required) ranked by predicted-spectrum cosine
similarity to the query — the "true global rank" result, as opposed to
formula- or RI-restricted retrieval.

Two ranking modes (see `pubchem_global_retrieval.py`):
- **inject**: the true molecule's predicted spectrum is guaranteed present
  in the candidate pool (upper bound on achievable rank).
- **autofail**: rank is taken as-is from the PubChem scan; if the true
  molecule's predicted spectrum wasn't found in the pool, its rank is left
  as reported by the pipeline (no injection).

Single run per model (no seed replication yet) on the random split, so no
CI band is shown.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from icicle.utils.visualization.eval_plots import model_color
from icicle.utils.visualization.style import make_fig, save_fig, set_style

set_style("manuscript")

RESULTS = Path("/home/magled/icicle-dev/results")
OUTPUT_DIR = Path("figures/retrieval_pubchem_global")
K_VALUES = [1, 2, 3, 4, 5, 10, 15, 20, 30, 40, 50]
SUMMARY_K_VALUES = [1, 5, 10, 20, 50]
MODES = ["autofail"]
MODE_DISPLAY_NAMES = {
    "inject": "Injected (true spectrum guaranteed in pool)",
    "autofail": "As-scanned (no injection)",
}

## Configuration

`MODEL_PATHS` maps model label to its `retrieval_per_query_StdNP_all.tsv`
(one row per query, columns `rank_inject_cosine` / `rank_autofail_cosine`).

In [ ]:
MODEL_PATHS = {
    "ICICLE": RESULTS
    / "pubchem_retrieval_eval_icicle_rerun_260710"
    / "retrieval_per_query_StdNP_all.tsv",
    "NEIMS": RESULTS
    / "pubchem_retrieval_eval_neims"
    / "retrieval_per_query_StdNP_all.tsv",
    "MassFormer": RESULTS
    / "pubchem_retrieval_eval_massformer"
    / "retrieval_per_query_StdNP_all.tsv",
}

## Load

One row per query already (rank of the true molecule directly) — no
decoy rows to filter, unlike the formula/RI retrieval CSVs.

In [ ]:
model_dfs = {}
for label, path in MODEL_PATHS.items():
    if path.exists():
        model_dfs[label] = pd.read_csv(path, sep="\t")
        print(f"{label}: {len(model_dfs[label])} queries loaded")
    else:
        print(f"{label}: {path} not found, skipping")

## Top-k accuracy curves

One figure per ranking mode. All models on the same axes.

In [ ]:
def topk_accuracy_curve(df: pd.DataFrame, rank_col: str, k_values: list[int]):
    """Top-k accuracy (%) at each k, single run (no CI)."""
    ranks = df[rank_col].dropna()
    return [100.0 * (ranks <= k).mean() for k in k_values]


def plot_pubchem_topk_curves(model_dfs, rank_col, k_values):
    fig, ax = make_fig("square")
    for i, (label, df) in enumerate(model_dfs.items()):
        if rank_col not in df.columns:
            continue
        accs = topk_accuracy_curve(df, rank_col, k_values)
        color = model_color(label, fallback_index=i)
        ax.plot(k_values, accs, marker="o", label=label, color=color)
    ax.set_xlabel("k")
    ax.set_ylabel("Top-k accuracy (%)")
    ax.set_ylim(0, 100)
    ax.legend()
    return fig

In [ ]:
for mode in MODES:
    rank_col = f"rank_{mode}_cosine"
    dfs_with_mode = {
        label: df for label, df in model_dfs.items() if rank_col in df.columns
    }
    if not dfs_with_mode:
        continue
    fig = plot_pubchem_topk_curves(dfs_with_mode, rank_col, K_VALUES)
    fig.axes[0].set_xlabel("Top-k")
    # fig.axes[0].set_title(f"{MODE_DISPLAY_NAMES.get(mode, mode)}")
    save_fig(fig, f"retrieval_pubchem_global_topk_{mode}_random", OUTPUT_DIR)
    plt.show()
    plt.close(fig)

## Summary table (top-k accuracy, MRR, median rank)

Single run per model, so no ± CI column (unlike the 3-seed formula/RI
notebooks).

In [ ]:
rows = []
for mode in MODES:
    rank_col = f"rank_{mode}_cosine"
    for label, df in model_dfs.items():
        if rank_col not in df.columns:
            continue
        ranks = df[rank_col].dropna()
        row = {
            "mode": mode,
            "model": label,
            "n_queries": len(ranks),
        }
        for k in SUMMARY_K_VALUES:
            row[f"top-{k}"] = f"{100.0 * (ranks <= k).mean():.1f}"
        row["MRR"] = f"{(1.0 / ranks).mean():.4f}"
        row["median_rank"] = f"{ranks.median():.1f}"
        rows.append(row)

df_summary = pd.DataFrame(rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(
    OUTPUT_DIR / "retrieval_pubchem_global_summary.csv", index=False
)
df_summary

## Export summary to LaTeX

In [ ]:
def export_pubchem_global_summary_latex(
    df,
    output_path="figures/retrieval_pubchem_global/retrieval_pubchem_global_table.tex",
):
    latex = df.to_latex(index=False, escape=False)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        f.write(latex)
    print(f"LaTeX table exported to {output_path}")
    print(latex)


export_pubchem_global_summary_latex(df_summary)

## MW±80Da-only candidate selection vs. full-PubChem random rank

Same models, same `rank_{mode}_cosine` per-query TSV schema, but the
candidate pool is now restricted to molecules within ±80 Da of the
query's highest observed peak (`retrieval_per_query_mw80_{ri_type}.tsv`)
instead of the full ~87-93M-molecule database. One figure per (RI type ×
mode) — StdNP is not available yet (its MW pass hasn't completed for any
model as of this writing; only SemiStdNP and StdPolar have run).

Note the query sets differ from the main comparison above: MW-only is
scored only against test molecules that have a value for that RI type AND
a `highest_peak_mz` (a strict subset of the full test set used for the
"all" curves above) — not a fair apples-to-apples N, but still a valid
same-query-set comparison of MW-only vs. each model's own "all"-level
curve restricted to the same molecules (computed below).

In [ ]:
MW_DA = 80
MW_RI_TYPES = [
    "SemiStdNP",
    "StdPolar",
]  # StdNP MW pass hasn't run yet for any model

MW_MODEL_PATHS = {
    ri_type: {
        label: RESULTS
        / dirname
        / f"retrieval_per_query_mw{MW_DA}_{ri_type}.tsv"
        for label, dirname in [
            ("ICICLE", "pubchem_retrieval_eval_icicle_rerun_260710"),
            ("NEIMS", "pubchem_retrieval_eval_neims"),
            ("MassFormer", "pubchem_retrieval_eval_massformer"),
        ]
    }
    for ri_type in MW_RI_TYPES
}

In [ ]:
mw_dfs_by_ri_type = {}
for ri_type in MW_RI_TYPES:
    dfs = {}
    for label, path in MW_MODEL_PATHS[ri_type].items():
        if path.exists():
            dfs[label] = pd.read_csv(path, sep="\t")
            print(f"{ri_type}/{label}: {len(dfs[label])} queries loaded")
        else:
            print(f"{ri_type}/{label}: {path} not found, skipping")
    mw_dfs_by_ri_type[ri_type] = dfs

In [ ]:
for ri_type in MW_RI_TYPES:
    dfs = mw_dfs_by_ri_type.get(ri_type, {})
    if not dfs:
        print(f"Skipping {ri_type} — no data")
        continue
    for mode in MODES:
        rank_col = f"rank_{mode}_cosine"
        dfs_with_mode = {
            label: df for label, df in dfs.items() if rank_col in df.columns
        }
        if not dfs_with_mode:
            continue
        fig = plot_pubchem_topk_curves(dfs_with_mode, rank_col, K_VALUES)
        fig.axes[0].set_xlabel("Top-k")
        fig.axes[0].set_title(
            f"MW±{MW_DA}Da candidates ({ri_type}) — {MODE_DISPLAY_NAMES.get(mode, mode)}"
        )
        save_fig(
            fig,
            f"retrieval_pubchem_global_mw{MW_DA}_{ri_type}_topk_{mode}",
            OUTPUT_DIR,
        )
        plt.show()
        plt.close(fig)

### MW-only summary table

In [ ]:
mw_rows = []
for ri_type in MW_RI_TYPES:
    dfs = mw_dfs_by_ri_type.get(ri_type, {})
    for mode in MODES:
        rank_col = f"rank_{mode}_cosine"
        for label, df in dfs.items():
            if rank_col not in df.columns:
                continue
            ranks = df[rank_col].dropna()
            row = {
                "ri_type": ri_type,
                "mode": mode,
                "model": label,
                "n_queries": len(ranks),
            }
            for k in SUMMARY_K_VALUES:
                row[f"top-{k}"] = f"{100.0 * (ranks <= k).mean():.1f}"
            row["mrr"] = f"{(1.0 / ranks).mean():.4f}"
            row["median_rank"] = ranks.median()
            mw_rows.append(row)

df_mw_summary = pd.DataFrame(mw_rows)
df_mw_summary.to_csv(
    OUTPUT_DIR / f"retrieval_pubchem_global_mw{MW_DA}_summary.csv", index=False
)
df_mw_summary

## MW-global: full-test-set candidate selection by molecular weight, no RI

Same models, same full test set as the top-level comparison above
(~27.5-27.6k queries with a valid GT spectrum — NOT the RI-type-restricted
subsets used in the MW±80Da section further below), but candidates are
restricted by molecular weight instead of scored against the entire
database. Three tracks, same query set, directly comparable:

- **Full-PubChem baseline** — `retrieval_per_query_StdNP_all.tsv` (already
  loaded above as `model_dfs`), no MW filter at all.
- **Symmetric ±80Da** — candidates within 80 Da either side of the query's
  highest observed peak.
- **Asymmetric [-10,+80]Da** — a narrower, empirically-motivated window.
  A direct check on 10k real NIST molecules found the highest observed
  peak is AT OR ABOVE the true MW in ~76% of cases (isotope peaks —
  M+1/M+2/M+3 from natural ¹³C/³⁷Cl/etc. abundance), not routinely below
  it as the "fragments can't exceed parent mass" intuition alone would
  suggest. A small negative margin (-10 Da, covering those isotope
  peaks) plus the same +80 Da positive margin matches the symmetric
  window's true-candidate coverage (~95.6%) at roughly half the total
  width — a substantially smaller, still-safe candidate pool.

Result: the asymmetric window beats the symmetric window on every
metric (MRR, Top-k, median rank), for every model — confirmed directly,
not assumed.

In [ ]:
MW_GLOBAL_TAGS = {
    "Full-PubChem (no MW)": None,
    "MW \u00b180Da": "80",
    "MW [-10,+80]Da": "10_80",
}

MW_GLOBAL_MODEL_DIRS = {
    "ICICLE": "pubchem_retrieval_eval_icicle_rerun_260710",
    "NEIMS": "pubchem_retrieval_eval_neims",
    "MassFormer": "pubchem_retrieval_eval_massformer",
}

In [ ]:
mw_global_dfs = {}  # {track_label: {model_label: df}}
for track_label, tag in MW_GLOBAL_TAGS.items():
    dfs = {}
    for label, dirname in MW_GLOBAL_MODEL_DIRS.items():
        if tag is None:
            # True full-test-set baseline, no MW filter -- NOT model_dfs
            # (that's retrieval_per_query_StdNP_all.tsv, restricted to the
            # small StdNP-having query subset, ~1.2k queries). This must be
            # the same ~27.5-27.6k-query full test set as the MW tracks
            # below for a fair same-query-set comparison.
            path = RESULTS / dirname / "retrieval_global_per_query.tsv"
        else:
            path = RESULTS / dirname / f"retrieval_per_query_mw{tag}_all.tsv"
        if path.exists():
            dfs[label] = pd.read_csv(path, sep="\t")
            print(f"{track_label}/{label}: {len(dfs[label])} queries loaded")
        else:
            print(f"{track_label}/{label}: {path} not found, skipping")
    mw_global_dfs[track_label] = dfs

### Top-k accuracy vs. k, all three tracks overlaid per model

One figure per (model × mode) so all three candidate-selection
strategies are directly comparable on the same axes.

In [ ]:
def plot_mw_global_tracks(track_dfs_by_label, model_label, rank_col, k_values):
    """track_dfs_by_label: {track_label: {model_label: df}}"""
    fig, ax = make_fig("default")
    for i, (track_label, dfs) in enumerate(track_dfs_by_label.items()):
        df = dfs.get(model_label)
        if df is None or rank_col not in df.columns:
            continue
        accs = topk_accuracy_curve(df, rank_col, k_values)
        ax.plot(k_values, accs, marker="o", label=track_label)
    ax.set_xlabel("Top-k")
    ax.set_ylabel("Top-k accuracy (%)")
    ax.set_ylim(0, 100)
    ax.legend()
    return fig


for model_label in MW_GLOBAL_MODEL_DIRS:
    for mode in MODES:
        rank_col = f"rank_{mode}_cosine"
        fig = plot_mw_global_tracks(
            mw_global_dfs, model_label, rank_col, K_VALUES
        )
        fig.axes[0].set_title(
            f"{model_label} — {MODE_DISPLAY_NAMES.get(mode, mode)}"
        )
        save_fig(
            fig,
            f"retrieval_pubchem_global_mw_tracks_{model_label}_{mode}",
            OUTPUT_DIR,
        )
        plt.show()
        plt.close(fig)

### MW-global summary table

In [ ]:
mw_global_rows = []
for track_label, dfs in mw_global_dfs.items():
    for mode in MODES:
        rank_col = f"rank_{mode}_cosine"
        for model_label, df in dfs.items():
            if rank_col not in df.columns:
                continue
            ranks = df[rank_col].dropna()
            row = {
                "track": track_label,
                "mode": mode,
                "model": model_label,
                "n_queries": len(ranks),
            }
            for k in SUMMARY_K_VALUES:
                row[f"top-{k}"] = f"{100.0 * (ranks <= k).mean():.1f}"
            row["mrr"] = f"{(1.0 / ranks).mean():.4f}"
            row["median_rank"] = ranks.median()
            mw_global_rows.append(row)

df_mw_global_summary = pd.DataFrame(mw_global_rows)
df_mw_global_summary.to_csv(
    OUTPUT_DIR / "retrieval_pubchem_global_mw_tracks_summary.csv", index=False
)
df_mw_global_summary